> ⚠️ Hinweis: Falls die Formeln nicht korrekt angezeigt werden, bitte in JupyterLab öffnen.

# Theorie

Ein periodisches Signal kann als Fourier-Reihe dargestellt werden:

$$
x(t) = \sum_{n=0}^{\infty} A_n \cos(2\pi n f_0 t + \phi_n)
$$

mit der Grundfrequenz:

$$
f_0 = \frac{1}{T_{\text{word}}}
$$

Ein idealer Tiefpass beschränkt die übertragbaren Frequenzen:

$$
H(f) =
\begin{cases}
1 & |f| \le f_c \\
0 & |f| > f_c
\end{cases}
$$

In dieser Simulation wird die Grenzfrequenz definiert als:

$$
f_c = N \cdot f_0
$$

Es werden also genau $N$ Harmonische übertragen.

 # Experiment

Variieren Sie die Anzahl der Harmonischen $N$ und beobachten Sie:

- Wie verändert sich die Signalform?
- Welche Frequenzen werden entfernt?
- Ab wann ist das Signal wieder gut rekonstruierbar?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

def simulate_harmonics(N=5):
    # ----- parameters -----
    bits = np.array([0, 1, 1, 0, 0, 0, 1, 0])
    Rb = 1000
    Tb = 1 / Rb
    fs = 200000
    A = 1.0

    samples_per_bit = int(fs / Rb)
    pattern = np.repeat(bits, samples_per_bit) * A

    num_repeats = 16
    x = np.tile(pattern, num_repeats)
    t = np.arange(len(x)) / fs

    # ----- fundamental frequency -----
    T_word = len(bits) * Tb
    f0 = 1 / T_word
    fc = N * f0   # cutoff based on harmonic number

    # ----- FFT -----
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(len(x), d=1/fs)

    # ideal low-pass
    H = (f <= fc).astype(float)
    Y = X * H
    y = np.fft.irfft(Y, n=len(x))

    # ----- harmonic extraction -----
    max_harm = 15
    harmonics = np.arange(1, max_harm + 1)

    Xmag = np.abs(X) / len(x)
    Ymag = np.abs(Y) / len(x)

    Xh = []
    Yh = []

    for n in harmonics:
        fh = n * f0
        idx = np.argmin(np.abs(f - fh))
        Xh.append(Xmag[idx])
        Yh.append(Ymag[idx])

    Xh = np.array(Xh)
    Yh = np.array(Yh)

    # ----- plots -----
    plt.figure(figsize=(14, 8))

    # ----- TOP: time domain -----
    plt.subplot(2, 1, 1)
    
    n_word = len(pattern)
    plt.plot(t[:n_word]*1e3, x[:n_word], label="Eingang")
    plt.plot(t[:n_word]*1e3, y[:n_word], label="Ausgang")
    
    plt.title(f"Zeitbereich (N = {N}, fc ≈ {fc:.0f} Hz)")
    plt.xlabel("Zeit [ms]")
    plt.ylabel("Amplitude")
    plt.ylim(-0.2, 1.2)
    plt.grid(True)
    plt.legend()
    
    # ----- BOTTOM LEFT: spectrum -----
    plt.subplot(2, 2, 3)
    
    plt.plot(f, Xmag, label="Eingang")
    plt.plot(f, Ymag, label="Ausgang")
    plt.axvline(fc, linestyle="--", label="cutoff")
    
    plt.xlim(0, 5000)
    plt.xlabel("Frequenz [Hz]")
    plt.ylabel("Amplitude")
    plt.title("Spektrum")
    plt.grid(True)
    plt.legend()
    
    # ----- BOTTOM RIGHT: harmonics -----
    plt.subplot(2, 2, 4)
    
    width = 0.35
    plt.bar(harmonics - width/2, Xh, width, label="Eingang")
    plt.bar(harmonics + width/2, Yh, width, label="Ausgang")
    
    plt.axvline(N + 0.5, linestyle="--", color='k')
    
    plt.xlabel("Vielfache der Grundfrequenz")
    plt.ylabel("Amplitude")
    plt.title("Harmonische")
    plt.grid(True)
    plt.legend()
    
    plt.tight_layout()
    plt.show()



interact(
    simulate_harmonics,
    N=IntSlider(value=5, min=1, max=20, step=1, description="Harmonische (Cutoff)")
);

 # Interpretation

- Bandbreitenbegrenzung entfernt hohe Frequenzen
  - → Flanken werden „weich“
  - → Bits überlappen sich (ISI, Inter Symbol Interference)

⚠️ Hinweise
- DC-Komponente (f = 0) wird absichtlich nicht dargestellt (Zu groß, lässt übrige Harmonische untergehen)
- Idealer Tiefpass ist physikalisch nicht realisierbar
- Reale Kabel wirken wie reale Tiefpässe (kein hartes Abschneiden von Harmonischen)
 
👉 Digitale Signale benötigen hohe Frequenzen für scharfe Flanken.